In [1]:
"""
Agentic Legal Search System - LOCAL VERSION (No API Keys, No Ollama)
Uses Legal-BERT + Hugging Face LLM directly

Installation:
"""

!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
from rank_bm25 import BM25Okapi

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━

2025-11-15 05:59:47.852278: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763186388.037334      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763186388.090799      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class LegalSearchAgent:
    def __init__(self, pdf_folder: str, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode

        self.vectorstore = None

        # BM25 structures
        self.bm25 = None
        self.bm25_corpus: List[List[str]] = []
        self.bm25_docs: List[object] = []           # stores langchain Document objects
        self.id_to_doc: Dict[str, object] = {}     # chunk_id -> Document

        # Load LLM (optional)
        print("🤖 Loading local LLM (Phi-2)...")
        try:
            model_name = "microsoft/phi-2"
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95
            )
            self.llm = HuggingFacePipeline(pipeline=pipe)
            print("Local LLM loaded successfully.")
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Falling back to extractive answers only.")
            self.llm = None

        # Embeddings
        print("📚 Loading Legal-BERT embeddings...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="nlpaueb/legal-bert-base-uncased",
            model_kwargs={'device': 'cuda'}
        )
        print("Legal-BERT loaded.\n")

    # ------------------------------
    # Build / load DB
    # ------------------------------
    def build_vectordb(self, force_rebuild: bool = False):
        """
        Build Chroma DB + BM25 index. Set force_rebuild=True to delete/overwrite existing DB.
        """
        if os.path.exists(self.db_path) and not force_rebuild:
            print("📚 Loading existing Chroma DB...")
            self.vectorstore = Chroma(
                persist_directory=self.db_path,
                embedding_function=self.embeddings
            )
            try:
                count = self.vectorstore._collection.count()
            except Exception:
                count = None
            print(f"Loaded existing DB. Vector count: {count}")
            # we still try to load BM25 if available - but best to rebuild if you suspect corruption
            return

        if os.path.exists(self.db_path) and force_rebuild:
            print("🧹 Force rebuilding: removing existing DB directory...")
            try:
                import shutil
                shutil.rmtree(self.db_path)
            except Exception as e:
                print("Could not remove DB folder:", e)

        print("🔨 Building new vector DB...")

        pdf_files = [f for f in os.listdir(self.pdf_folder) if f.lower().endswith(".pdf")]

        if self.test_mode:
            pdf_files = pdf_files[:5]
            print(f"⚠️ TEST MODE: Using only first {len(pdf_files)} PDFs")
        else:
            print(f"Found {len(pdf_files)} PDFs to process.")

        all_pages = []

        # LOAD PDFS
        for idx, pdf in enumerate(pdf_files, 1):
            pdf_path = os.path.join(self.pdf_folder, pdf)
            try:
                print(f"📄 Loading [{idx}/{len(pdf_files)}]: {pdf}")
                loader = PyPDFLoader(pdf_path)
                pages = loader.load()
                base = pdf.replace(".pdf", "")
                parts = re.split(r"[_\-]", base)
                case_type = "_".join(parts[:-1]) if len(parts) >= 2 else base
                case_year = parts[-1] if parts[-1].isdigit() else "unknown"

                for page in pages:
                    # assign unique chunk id
                    chunk_id = str(uuid.uuid4())
                    page.metadata["source_file"] = pdf
                    page.metadata["case_number"] = base
                    page.metadata["case_year"] = case_year
                    page.metadata["case_type"] = case_type
                    page.metadata["chunk_id"] = chunk_id

                    # skip empty pages
                    if not page.page_content or not page.page_content.strip():
                        continue

                all_pages.extend(pages)

            except Exception as e:
                print(f"❌ Error loading {pdf_path}: {e}")

        print(f"Loaded pages before splitting: {len(all_pages)}")

        # SPLIT
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )
        chunks = splitter.split_documents(all_pages)
        # assign chunk_id (split_documents may create new Document objects; ensure they have chunk_id)
        for ch in chunks:
            if "chunk_id" not in ch.metadata:
                ch.metadata["chunk_id"] = str(uuid.uuid4())
            # normalize source_file
            ch.metadata["source_file"] = ch.metadata.get("source_file", "unknown")
            # remove empty chunks
        chunks = [c for c in chunks if c.page_content and c.page_content.strip()]
        print(f"✂️ Split into {len(chunks)} non-empty chunks.")

        # BUILD CHROMA DB
        print("🔮 Generating embeddings and saving to Chroma (may take a while)...")
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_path
        )
        try:
            count = self.vectorstore._collection.count()
        except Exception:
            count = None
        print(f"✅ Saved Chroma DB at {self.db_path}. Vector count (approx): {count}")

        # BUILD BM25 CORPUS AND MAPPINGS
        print("Building BM25 index and internal mappings...")
        self.bm25_corpus = []
        self.bm25_docs = []
        self.id_to_doc = {}

        for ch in chunks:
            tokens = self._simple_tokenize(ch.page_content)
            if len(tokens) == 0:
                continue
            self.bm25_corpus.append(tokens)
            self.bm25_docs.append(ch)
            self.id_to_doc[ch.metadata["chunk_id"]] = ch

        if len(self.bm25_corpus) == 0:
            raise RuntimeError("BM25 corpus empty after indexing — check PDF extraction / OCR")

        self.bm25 = BM25Okapi(self.bm25_corpus)
        print("📌 BM25 index ready.\n")

        # DIAGNOSTICS: print a sample of metadata & docs to verify correctness
        print("Diagnostic sample (first 5 chunk metadata):")
        sample_ids = list(self.id_to_doc.keys())[:5]
        for cid in sample_ids:
            doc = self.id_to_doc[cid]
            print(" - chunk_id:", cid, "source:", doc.metadata.get("source_file"), "len:", len(doc.page_content))

    # ------------------------------
    # Utilities
    # ------------------------------
    @staticmethod
    def _simple_tokenize(text: str) -> List[str]:
        # simple whitespace/token split; you can replace with better tokenizer if needed
        return [t.strip() for t in re.split(r"\s+", text) if t.strip()]

    def _detect_case_number(self, query: str) -> str | None:
        """
        Detect patterns like: CPLA 210 of 2010, C.A.981_2018, Cr.A 52/2021
        Normalized with underscores: e.g., "CPLA_210_2010"
        """
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            # keep file extension not added here; we compare to source_file directly
            return case
        return None

    # ------------------------------
    # HYBRID RETRIEVER
    # ------------------------------
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Dict]:
        """
        Hybrid retrieval:
         - if case-number detected: try metadata-filtered semantic search first (high precision)
         - else: do semantic (embedding) + BM25 and merge using chunk_id
        """
        results_map: Dict[str, float] = {}

        # 1) case number exact-match boost / filter
        case_num = self._detect_case_number(query)
        if case_num:
            # try metadata-filtered semantic search first
            filter_dict = {"source_file": f"{case_num}.pdf"}
            try:
                semantic_filtered = self.vectorstore.similarity_search_with_score(query, k=k, filter=filter_dict)
            except Exception:
                semantic_filtered = []
            if semantic_filtered:
                # use these top results immediately (they're targeted)
                out = []
                for doc, score in semantic_filtered[:k]:
                    chunk_id = doc.metadata.get("chunk_id")
                    out.append({
                        "content": doc.page_content,
                        "source": doc.metadata.get("source_file", "unknown"),
                        "case_number": doc.metadata.get("case_number", "unknown"),
                        "case_year": doc.metadata.get("case_year", "unknown"),
                        "case_type": doc.metadata.get("case_type", "unknown"),
                        "chunk_id": chunk_id,
                        "score": float(1 - score)
                    })
                return out

        # 2) semantic search (top 2k to give BM25 room)
        try:
            semantic_hits = self.vectorstore.similarity_search_with_score(query, k=2 * k)
        except Exception as e:
            print("Semantic search failed:", e)
            semantic_hits = []

        # accumulate semantic scores keyed by chunk_id
        for doc, s_score in semantic_hits:
            cid = doc.metadata.get("chunk_id")
            if not cid:
                continue
            # convert chroma distance-like score to similarity
            sem_sim = (1.0 - float(s_score))
            results_map[cid] = results_map.get(cid, 0.0) + 0.6 * sem_sim

        # 3) BM25 search
        tokens = self._simple_tokenize(query)
        bm25_scores = self.bm25.get_scores(tokens)
        top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:2 * k]
        for idx in top_idx:
            ch = self.bm25_docs[idx]
            cid = ch.metadata.get("chunk_id")
            if not cid:
                continue
            results_map[cid] = results_map.get(cid, 0.0) + 0.4 * float(bm25_scores[idx])

        # 4) prepare ranked list
        ranked = sorted(results_map.items(), key=lambda x: x[1], reverse=True)[:k]
        out = []
        for cid, combined_score in ranked:
            doc = self.id_to_doc.get(cid)
            if not doc:
                continue
            out.append({
                "content": doc.page_content,
                "source": doc.metadata.get("source_file", "unknown"),
                "case_number": doc.metadata.get("case_number", "unknown"),
                "case_year": doc.metadata.get("case_year", "unknown"),
                "case_type": doc.metadata.get("case_type", "unknown"),
                "chunk_id": cid,
                "score": float(combined_score)
            })
        return out

    # ------------------------------
    # Answer generator
    # ------------------------------
    def _generate_answer(self, query: str, docs: List[Dict]) -> str:
        if not self.llm:
            return self._simple_answer(docs)

        context = "\n\n".join(f"[{d['source']}] {d['content'][:500]}" for d in docs[:5])
        prompt = f"""
You are a legal AI assistant summarizing Pakistan Supreme Court judgments.
Use ONLY the excerpts below. NEVER guess or invent.

QUERY: {query}

EXCERPTS:
{context}

Give a concise answer based strictly on the excerpts.
"""
        try:
            out = self.llm.invoke(prompt)
            return out if isinstance(out, str) else str(out)
        except Exception as e:
            print("LLM generation failed:", e)
            return self._simple_answer(docs)

    def _simple_answer(self, docs: List[Dict]):
        ans = "Based on retrieved excerpts:\n\n"
        for d in docs[:5]:
            ans += f"{d['source']} → {d['content'][:300]}\n\n"
        return ans

    # ------------------------------
    # Main search API
    # ------------------------------
    def search(self, query: str, k: int = 10) -> List[Document]:
            print(f"\n🔍 QUERY: {query}")
    
            case_num = self._detect_case_number(query)
            if case_num:
                print(f"📋 Detected case number: {case_num}")
                # Get more results to filter from
                results = self._retrieve_documents(query, k=k*3)
                # Filter to exact case match first
                exact_match = [r for r in results if case_num.lower() in r.metadata['case_number'].lower()]
                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match: {case_num}")
                else:
                    print(f"⚠️ No exact case found, returning semantic matches")
                    results = results[:k]
            else:
                results = self._retrieve_documents(query, k=k)
    
            print("\n📄 Retrieved PDFs:")
            for r in results:
                print(" -", r.metadata.get("source_file", "unknown"))
    
            return results

In [3]:
PDF_FOLDER = "/kaggle/input/fyp-data/supreme_court_judgments"

print("Starting LOCAL Legal Search Agent\n")
print("No API keys, No Ollama - runs 100% locally!\n")

agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)

agent.build_vectordb()

print("\n" + "="*80)
print("LEGAL SEARCH AGENT READY (LOCAL)")
print("="*80)


Starting LOCAL Legal Search Agent

No API keys, No Ollama - runs 100% locally!

🤖 Loading local LLM (Phi-2)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

📄 Loading [751/975]: Crl.P.L.A.887-L_2013.pdf
📄 Loading [752/975]: C.A.377_2014.pdf
📄 Loading [753/975]: C.P.L.A.3062_2022.pdf
📄 Loading [754/975]: J.P.195_2017.pdf
📄 Loading [755/975]: C.M.A.12587_2021.pdf
📄 Loading [756/975]: C.P.L.A.546_2021.pdf
📄 Loading [757/975]: C.M.Appeal.39_2021.pdf
📄 Loading [758/975]: C.P.L.A.3300_2024.pdf
📄 Loading [759/975]: Crl.P.L.A.952_2021.pdf
📄 Loading [760/975]: Crl.P.L.A.1602_2023.pdf
📄 Loading [761/975]: Crl.P.L.A.668_2019.pdf
📄 Loading [762/975]: J.P.50_2023.pdf
📄 Loading [763/975]: J.P.252_2020.pdf
📄 Loading [764/975]: C.A.647_2018.pdf
📄 Loading [765/975]: Crl.A.379_2021.pdf
📄 Loading [766/975]: C.P.L.A.159_2021.pdf
📄 Loading [767/975]: C.P.L.A.394-P_2010.pdf
📄 Loading [768/975]: C.P.L.A.559-P_2024.pdf
📄 Loading [769/975]: C.P.L.A.1692-L_2020.pdf
📄 Loading [770/975]: Crl.P.L.A.1075-L_2020.pdf
📄 Loading [771/975]: Crl.P.L.A.69-Q_2022.pdf
📄 Loading [772/975]: C.P.L.A.3179-L_2023.pdf
📄 Loading [773/975]: C.A.17-Q_2023.pdf
📄 Loading [774/975]: S.M.C.

📄 Loading [776/975]: C.P.L.A.3644_2020.pdf
📄 Loading [777/975]: C.P.L.A.1290-L_2019.pdf
📄 Loading [778/975]: C.A.1414_2013.pdf
📄 Loading [779/975]: C.P.L.A.3984_2024.pdf
📄 Loading [780/975]: C.P.L.A.109-L_2024.pdf
📄 Loading [781/975]: Crl.P.L.A.231_2021.pdf
📄 Loading [782/975]: Crl.P.L.A.1690-L_2016.pdf
📄 Loading [783/975]: C.P.L.A.2987-L_2019.pdf
📄 Loading [784/975]: J.P.516_2018.pdf
📄 Loading [785/975]: Crl.A.91_2024.pdf
📄 Loading [786/975]: C.P.L.A.2537_2020.pdf
📄 Loading [787/975]: Crl.P.L.A.146_2025.pdf
📄 Loading [788/975]: Crl.A.238_2021.pdf
📄 Loading [789/975]: C.A.1444_2013.pdf
📄 Loading [790/975]: C.A.700_2014.pdf
📄 Loading [791/975]: C.A.1692_2021.pdf
📄 Loading [792/975]: C.P.L.A.2790_2018.pdf
📄 Loading [793/975]: Crl.P.L.A.1117_2024.pdf
📄 Loading [794/975]: C.P.L.A.4389_2023.pdf
📄 Loading [795/975]: C.P.L.A.414_2021.pdf
📄 Loading [796/975]: C.P.L.A.5666_2024.pdf
📄 Loading [797/975]: C.A.725_2008.pdf
📄 Loading [798/975]: C.A.875_2017.pdf
📄 Loading [799/975]: C.P.21_2022.pdf
📄

📄 Loading [830/975]: C.P.L.A.2743_2017.pdf
❌ Error loading /kaggle/input/fyp-data/supreme_court_judgments/C.P.L.A.2743_2017.pdf: Invalid Elementary Object starting with b'I' @22327: b'l)3123.3n)19( )98 0.IT\n/F2 12.0 Tf\n 0.0 0.0 rg\n0.9998 137(s)8( )-70(ne2(h)19(e)3'
📄 Loading [831/975]: C.P.L.A.5601_2021.pdf
📄 Loading [832/975]: C.A.364_2023.pdf
📄 Loading [833/975]: J.P.14_2020.pdf
📄 Loading [834/975]: C.P.L.A.1809_2020.pdf
📄 Loading [835/975]: Crl.A.525_2022.pdf
📄 Loading [836/975]: Crl.P.L.A.1016-L_2021.pdf
📄 Loading [837/975]: Crl.A.425_2019.pdf
📄 Loading [838/975]: Crl.A.36_2023.pdf
📄 Loading [839/975]: C.P.L.A.1842-L_2022.pdf
📄 Loading [840/975]: C.A.1518_2013.pdf
📄 Loading [841/975]: J.P.42_2017.pdf
📄 Loading [842/975]: Crl.A.199_2023.pdf
📄 Loading [843/975]: Crl.A.322_2018.pdf
📄 Loading [844/975]: C.P.L.A.1618_2024.pdf
📄 Loading [845/975]: C.A.248_2014.pdf
📄 Loading [846/975]: C.P.6_2023.pdf
📄 Loading [847/975]: C.A.138-L_2010.pdf
📄 Loading [848/975]: Crl.P.L.A.54_2023.pdf
📄 L

📄 Loading [873/975]: C.P.L.A.1926-L_2015.pdf
📄 Loading [874/975]: C.P.L.A.1010-L_2022.pdf
📄 Loading [875/975]: C.A.550-L_2009.pdf
📄 Loading [876/975]: C.A.151-P_2013.pdf
📄 Loading [877/975]: J.P.23_2023.pdf
📄 Loading [878/975]: Crl.A.314-L_2020.pdf
📄 Loading [879/975]: C.A.1683_2014.pdf
📄 Loading [880/975]: Crl.P.L.A.1124-L_2015.pdf
📄 Loading [881/975]: C.P.L.A.184_2024.pdf
📄 Loading [882/975]: C.A.1227_2016.pdf
📄 Loading [883/975]: C.P.L.A.1737-L_2020.pdf
📄 Loading [884/975]: C.A.350_2020.pdf
📄 Loading [885/975]: C.A.1394_2024.pdf
📄 Loading [886/975]: C.P.L.A.1422-L_2021.pdf
📄 Loading [887/975]: C.P.L.A.2478_2024.pdf
📄 Loading [888/975]: Crl.P.L.A.150-K_2024.pdf
📄 Loading [889/975]: Crl.P.L.A.435_2021.pdf
📄 Loading [890/975]: C.P.L.A.4177_2024.pdf
📄 Loading [891/975]: C.P.L.A.2475-L_2024.pdf
📄 Loading [892/975]: C.A.2434_2016.pdf
📄 Loading [893/975]: Crl.P.L.A.660_2024.pdf
📄 Loading [894/975]: C.A.700_2016.pdf
📄 Loading [895/975]: C.P.L.A.473-K_2023.pdf
📄 Loading [896/975]: C.P.L.A.11

📄 Loading [911/975]: Crl.A.81-L_2017.pdf
📄 Loading [912/975]: C.P.L.A.4618_2019.pdf
📄 Loading [913/975]: C.P.L.A.2414-L_2015.pdf
📄 Loading [914/975]: Crl.P.L.A.1288-L_2017.pdf
📄 Loading [915/975]: Crl.P.L.A.230_2019.pdf
📄 Loading [916/975]: J.P.611_2022.pdf
📄 Loading [917/975]: C.P.L.A.183_2024.pdf
📄 Loading [918/975]: C.M.A.3610_2022.pdf
📄 Loading [919/975]: C.R.P.255_2021.pdf
📄 Loading [920/975]: C.A.8-Q_2017.pdf
📄 Loading [921/975]: C.R.P.988_2023.pdf
📄 Loading [922/975]: J.P.644_2017.pdf
📄 Loading [923/975]: C.P.L.A.757-L_2021.pdf
📄 Loading [924/975]: C.P.L.A.3116_2022.pdf
📄 Loading [925/975]: C.A.139-P_2013.pdf
📄 Loading [926/975]: Crl.A.304_2020.pdf
📄 Loading [927/975]: Crl.P.L.A.806_2022.pdf
📄 Loading [928/975]: C.P.L.A.3127_2020.pdf
📄 Loading [929/975]: C.A.24-Q_2014.pdf
📄 Loading [930/975]: Crl.A.144-L_2020.pdf
📄 Loading [931/975]: Crl.A.92-L_2017.pdf
📄 Loading [932/975]: C.P.L.A.2400-L_2022.pdf
📄 Loading [933/975]: Crl.P.L.A.344_2018.pdf
📄 Loading [934/975]: C.P.L.A.1189_2025

In [ ]:
print("\nType your query (or 'quit' to exit)")
print("Example: 'Why did the court reject SIC's appeal?'\n")

while True:
    query = input("\n💬 Your query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    if not query:
        continue
    
    try:
        result = agent.search(query)
        
        print("\n📎 Relevant PDFs:")
        for pdf in result['relevant_pdfs']:
            print(f"   - {pdf}")
        print()
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Please try again or check your setup.\n")


Type your query (or 'quit' to exit)
Example: 'Why did the court reject SIC's appeal?'




💬 Your query:  What was CPLA 210 of 2024 about?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔍 QUERY: What was CPLA 210 of 2024 about?

📄 Retrieved:
 - C.P.L.A.3297_2024.pdf (chunk_id: fa70002a-13f5-4e79-984c-45310e8811f1 score: 13.2889 )
 - C.P.L.A.6-L_2023.pdf (chunk_id: fffc4ed5-2a0a-49fa-ad73-6b05eb178f28 score: 11.7006 )
 - C.A.1509_2021.pdf (chunk_id: 5dc408ae-764d-463c-8cd2-ff740c0bb52a score: 11.5349 )
 - C.P.L.A.1573_2024.pdf (chunk_id: 6e9adab5-481d-4348-b865-d7f37d0c310b score: 11.4661 )
 - C.P.L.A.1573_2024.pdf (chunk_id: d5872b8a-c9cd-4cbf-a573-dcf6dde524fb score: 11.1186 )
 - C.A.2026_2022.pdf (chunk_id: 9a80f18f-fff2-4bc2-8b9c-7103182f2938 score: 6.9637 )
 - C.P.L.A.3578_2024.pdf (chunk_id: 020e6f66-93a3-42b1-8cf8-96b924ecaab1 score: 6.7071 )
 - C.P.L.A.5516_2024.pdf (chunk_id: 3583a70a-16dd-489c-87e4-ca28ffd26757 score: 6.1225 )
 - .pdf (chunk_id: 22d37d75-88c0-44c9-9ccc-7a317ddb4e0e score: 5.9825 )
 - C.P.L.A.1618_2024.pdf (chunk_id: dfce8680-1398-43a4-a9ff-6b19b051954c score: 5.8777 )
 - .pdf (chunk_id: e0a3ab1e-0748-41f9-a272-63073bf10548 score: 5.7635 )
 -